# Phase 3 – File 5: Matrix & Vector Calculus (Jacobians, Hessians & VJPs)

**The exact mathematical machinery that powers modern Automatic Differentiation (`torch.autograd`) and deep neural network training.**

---

## 1. Why Matrix and Vector Calculus?

In basic calculus, we take derivatives of scalar functions with respect to a single variable: $\frac{df}{dx}$.
In Deep Learning, a single neural network layer transforms a **vector of inputs** $\mathbf{x} \in \mathbb{R}^n$ into a **vector of outputs** $\mathbf{y} \in \mathbb{R}^m$ via weight matrices $\mathbf{W} \in \mathbb{R}^{m \times n}$:
$$\mathbf{y} = \sigma(\mathbf{W}\mathbf{x} + \mathbf{b})$$

To train the network, we must calculate:
1. How a vector output changes with respect to a vector input: **The Jacobian Matrix $\mathbf{J}$**.
2. How the overall scalar loss curves in multi-dimensional parameter space: **The Hessian Matrix $\mathbf{H}$**.
3. How gradients flow backward efficiently through the computational graph: **Vector-Jacobian Products (VJPs)**.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

print("Numpy and Sympy loaded successfully!")


---

## 2. The Gradient Vector (Scalar-to-Vector Derivative)

For a scalar function of a vector $f(\mathbf{x}): \mathbb{R}^n \to \mathbb{R}$:
$$\nabla_{\mathbf{x}} f(\mathbf{x}) = \begin{bmatrix} \frac{\partial f}{\partial x_1} \\ \frac{\partial f}{\partial x_2} \\ \vdots \\ \frac{\partial f}{\partial x_n} \end{bmatrix} \in \mathbb{R}^n$$

### Common Vector Derivative Identities:
1. $\nabla_{\mathbf{x}} (\mathbf{a}^T \mathbf{x}) = \mathbf{a}$
2. $\nabla_{\mathbf{x}} (\mathbf{x}^T \mathbf{A} \mathbf{x}) = (\mathbf{A} + \mathbf{A}^T)\mathbf{x} \quad (\text{if } \mathbf{A} \text{ is symmetric, } = 2\mathbf{A}\mathbf{x})$
3. $\nabla_{\mathbf{x}} \|\mathbf{x}\|_2^2 = 2\mathbf{x}$


In [ ]:
# Verify gradient identity: f(x) = x^T A x
x1, x2 = sp.symbols('x1 x2')
x = sp.Matrix([x1, x2])
A = sp.Matrix([[3, 1], [1, 2]]) # Symmetric matrix

f = (x.T * A * x)[0]
print("f(x) =", f)

# Compute gradient symbolically
grad_f = sp.Matrix([sp.diff(f, x1), sp.diff(f, x2)])
print("Symbolic Gradient [df/dx1, df/dx2] =\n", grad_f)

# Identity: 2 * A * x
expected_grad = 2 * A * x
print("2 * A * x =\n", expected_grad)
assert grad_f == expected_grad
print("Identity verified!")


---

## 3. The Jacobian Matrix (Vector-to-Vector Derivative)

When a function maps an $n$-dimensional vector $\mathbf{x}$ to an $m$-dimensional vector $\mathbf{f}(\mathbf{x}) = [f_1(\mathbf{x}), f_2(\mathbf{x}), \dots, f_m(\mathbf{x})]^T$, its derivative is an $m \times n$ matrix called the **Jacobian**:

$$\mathbf{J} = \frac{\partial \mathbf{f}}{\partial \mathbf{x}} = \begin{bmatrix} 
\frac{\partial f_1}{\partial x_1} & \frac{\partial f_1}{\partial x_2} & \dots & \frac{\partial f_1}{\partial x_n} \\
\frac{\partial f_2}{\partial x_1} & \frac{\partial f_2}{\partial x_2} & \dots & \frac{\partial f_2}{\partial x_n} \\
\vdots & \vdots & \ddots & \vdots \\
\frac{\partial f_m}{\partial x_1} & \frac{\partial f_m}{\partial x_2} & \dots & \frac{\partial f_m}{\partial x_n}
\end{bmatrix} \in \mathbb{R}^{m \times n}$$

### Real AI/ML Example: The Jacobian of the Softmax Function
For logits $\mathbf{z} \in \mathbb{R}^K$, the softmax probability is $p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$.
Its Jacobian is:
$$\frac{\partial p_i}{\partial z_j} = \begin{cases} p_i(1 - p_i) & \text{if } i = j \\ -p_i p_j & \text{if } i \neq j \end{cases} = \text{diag}(\mathbf{p}) - \mathbf{p}\mathbf{p}^T$$


In [ ]:
def softmax(z):
    e_z = np.exp(z - np.max(z))
    return e_z / e_z.sum()

def softmax_jacobian(z):
    p = softmax(z)
    # J = diag(p) - p * p^T
    return np.diag(p) - np.outer(p, p)

z = np.array([2.0, 1.0, 0.1])
p = softmax(z)
J = softmax_jacobian(z)

print("Logits z:", z)
print("Softmax probabilities p:", p)
print("Softmax Jacobian Matrix J (3x3):\n", np.round(J, 4))
print("Row sum of Jacobian (must be 0 because sum(p)=1):", np.round(J.sum(axis=1), 4))


---

## 4. The Hessian Matrix (Second-Order Curvature)

For a scalar loss function $L(\mathbf{x}): \mathbb{R}^n \to \mathbb{R}$, the matrix of all second-order partial derivatives is the **Hessian Matrix $\mathbf{H}$**:

$$\mathbf{H} = \nabla^2 L(\mathbf{x}) = \begin{bmatrix}
\frac{\partial^2 L}{\partial x_1^2} & \frac{\partial^2 L}{\partial x_1 \partial x_2} & \dots & \frac{\partial^2 L}{\partial x_1 \partial x_n} \\
\frac{\partial^2 L}{\partial x_2 \partial x_1} & \frac{\partial^2 L}{\partial x_2^2} & \dots & \frac{\partial^2 L}{\partial x_2 \partial x_n} \\
\vdots & \vdots & \ddots & \vdots \\
\frac{\partial^2 L}{\partial x_n \partial x_1} & \frac{\partial^2 L}{\partial x_n \partial x_2} & \dots & \frac{\partial^2 L}{\partial x_n^2}
\end{bmatrix} \in \mathbb{R}^{n \times n}$$

### Eigenvalues of the Hessian and Optimization:
- **All $\lambda_i > 0$ ($\mathbf{H} \succ 0$)**: Local Minimum (convex bowl).
- **All $\lambda_i < 0$ ($\mathbf{H} \prec 0$)**: Local Maximum (concave dome).
- **Mixed signs ($\lambda_1 > 0, \lambda_2 < 0$)**: **Saddle Point** (very common in deep neural networks!).


In [ ]:
# Computing the Hessian of a 2D function: f(x, y) = x^3 - 3x*y^2 (Monkey Saddle)
x, y = sp.symbols('x y')
f = x**3 - 3*x*y**2

# First derivatives (Gradient)
fx = sp.diff(f, x)
fy = sp.diff(f, y)

# Second derivatives (Hessian)
H = sp.Matrix([
    [sp.diff(fx, x), sp.diff(fx, y)],
    [sp.diff(fy, x), sp.diff(fy, y)]
])

print("Function f(x,y) =", f)
print("Gradient:\n", sp.Matrix([fx, fy]))
print("Hessian Matrix:\n", H)


---

## 5. Vector-Jacobian Products (VJPs) — How Autograd Actually Works

In Deep Learning, neural networks have millions of parameters, and the output loss $L$ is a scalar.
Suppose an intermediate layer computes $\mathbf{y} = \mathbf{f}(\mathbf{x})$.
During backpropagation, we receive the gradient of the scalar loss with respect to $\mathbf{y}$:
$$\mathbf{v} = \frac{\partial L}{\partial \mathbf{y}} \in \mathbb{R}^{1 \times m}$$

To find $\frac{\partial L}{\partial \mathbf{x}}$, the chain rule says:
$$\frac{\partial L}{\partial \mathbf{x}} = \mathbf{v} \cdot \mathbf{J} \quad \text{where } \mathbf{J} = \frac{\partial \mathbf{y}}{\partial \mathbf{x}} \in \mathbb{R}^{m \times n}$$

### Why is this crucial?
PyTorch **never** constructs the massive $m \times n$ Jacobian matrix explicitly in memory! Instead, it computes the **Vector-Jacobian Product (VJP)** $\mathbf{v}^T \mathbf{J}$ directly in $O(n)$ time!


In [ ]:
# Demonstrating VJP in Python
# Layer: y = W x + b
W = np.random.randn(3, 4) # 3 outputs, 4 inputs
x = np.random.randn(4)
y = W @ x

# Upstream gradient from loss: dL/dy
v = np.random.randn(3)

# 1. Full Jacobian approach: J = W
J = W
dL_dx_full = v @ J

# 2. Direct VJP approach (W^T v)
dL_dx_vjp = W.T @ v

print("dL/dx via full Jacobian:\n", dL_dx_full)
print("dL/dx via VJP (W^T v):\n", dL_dx_vjp)
assert np.allclose(dL_dx_full, dL_dx_vjp)
print("VJP matches full chain rule perfectly with zero matrix allocation overhead!")


---

## 6. Summary & Key Takeaways

1. **Gradients** $\nabla f(\mathbf{x})$ map scalar outputs to vector inputs.
2. **Jacobians** $\mathbf{J} = \frac{\partial \mathbf{f}}{\partial \mathbf{x}}$ capture multi-input multi-output layer transformations.
3. **Hessians** $\mathbf{H} = \nabla^2 L(\mathbf{x})$ reveal second-order curvature, identifying local minima, maxima, and saddle points.
4. **Vector-Jacobian Products (VJPs)** enable reverse-mode automatic differentiation to compute backpropagation gradients in $O(1)$ passes without storing huge Jacobian matrices.
